# Iridium 1.0 -- start here: `iridium_studio.ipynb`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sporadicstudiosind-cloud/test/blob/claude/gallant-faraday-lhycva/notebooks/iridium_studio.ipynb)

This notebook (the old fixed single-preset Colab trainer) has been folded
into **`iridium_studio.ipynb`**, in this same `notebooks/` directory. That
notebook does everything this one used to -- detect the T4, pick a preset,
dry-run it, train in rounds, chat with the result -- and adds the private-
repo clone step, resume-after-disconnect instructions, and the optional
tool-use second stage.

Open `iridium_studio.ipynb` instead of this file; nothing below trains
anything.

In [1]:
# This repository is private until Iridium 1.0 ships. This cell never
# hard-codes or prints a token: it reads one from the host's own secret
# store (or an env var on a plain Jupyter host) and hands it to git only
# through an environment-scoped header, so it never lands in .git/config.
import os, sys, subprocess
from pathlib import Path
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    token = None

REPO_OWNER = 'sporadicstudiosind-cloud'
REPO_NAME = 'test'
RELEASE_BRANCH = 'claude/gallant-faraday-lhycva'
ROOT = Path.cwd()

# The token travels in an environment-scoped git config header: never in
# the URL (git would store it in .git/config) and never in the argv (a
# failed subprocess prints its argv).
env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
if token:
    import base64
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env.update(GIT_CONFIG_COUNT='1',
               GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
               GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {basic}')
    del basic

if (ROOT / 'iridium' / 'presets.py').is_file():
    print('Already inside a checkout of the repository:', ROOT)
else:
    ROOT = Path.cwd() / REPO_NAME
    if ROOT.is_dir():
        # A checkout left over from earlier in this session would
        # otherwise silently keep running whatever code it was cloned
        # with, including bugs fixed upstream since. Bring it up to date.
        # fetch + reset rather than pull --ff-only: a depth-1 clone has no
        # history for git to prove a fast-forward with.
        dirty = subprocess.run(['git', '-C', str(ROOT), 'status', '--porcelain',
                                '--untracked-files=no'],
                               capture_output=True, text=True).stdout.strip()
        if dirty:
            raise RuntimeError('The existing checkout at %s has local edits, so it was '
                               'not updated:\n%s\nDelete it (!rm -rf %s) and rerun this '
                               'cell for a fresh clone.' % (ROOT, dirty, ROOT))
        fetched = subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1',
                                  'origin', RELEASE_BRANCH],
                                 env=env, capture_output=True, text=True)
        if fetched.returncode != 0:
            raise RuntimeError('Could not update the existing checkout at %s (exit %d). '
                               'Delete it (!rm -rf %s) and rerun this cell.' %
                               (ROOT, fetched.returncode, ROOT))
        before = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'],
                                capture_output=True, text=True).stdout.strip()
        subprocess.run(['git', '-C', str(ROOT), 'reset', '--hard', 'FETCH_HEAD'],
                       check=True, capture_output=True)
        after = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'],
                               capture_output=True, text=True).stdout.strip()
        print('Updated existing checkout at', ROOT)
        if before != after and 'iridium' in sys.modules:
            # New code on disk does not replace modules already imported:
            # the old ones keep running and the update never executes.
            raise RuntimeError('The checkout was just updated, but this kernel already '
                               'imported the old code. Restart the kernel, then run the '
                               'notebook from the top.')
    else:
        if not token:
            print('No GITHUB_TOKEN found. In Kaggle: Add-ons, Secrets, add GITHUB_TOKEN with a fine-grained PAT that can read this repository, then attach it to this notebook.')
            print('Trying an unauthenticated clone (works only if the repo is public)...')
        done = subprocess.run(['git', 'clone', '--depth', '1', '--branch', RELEASE_BRANCH,
                               f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git',
                               str(ROOT)], env=env, capture_output=True, text=True)
        if done.returncode != 0:
            raise RuntimeError('git clone failed (exit %d). Check the token can read '
                               'the repository and the branch exists.' % done.returncode)
        print('Cloned', REPO_OWNER + '/' + REPO_NAME, '@', RELEASE_BRANCH, 'to', ROOT)
del env
revision = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()
print('source revision:', revision or 'unknown')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working directory:', Path.cwd())

No GITHUB_TOKEN found. In Kaggle: Add-ons, Secrets, add GITHUB_TOKEN with a fine-grained PAT that can read this repository, then attach it to this notebook.
Trying an unauthenticated clone (works only if the repo is public)...
Cloned sporadicstudiosind-cloud/test @ claude/gallant-faraday-lhycva to /content/test
source revision: 9009d4f
working directory: /content/test


In [2]:
!pip install -q -e ".[data]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for iridium (pyproject.toml) ... done


In [3]:
import time
SESSION_STARTED = time.monotonic()

import torch
from iridium.presets import get_preset, preset_table
from iridium.runtime.device import detect
from iridium.training.run_preset import dry_run

info = detect()
DEVICE = info.device
print(info.describe())
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print('GPU:', torch.cuda.get_device_name(0))

PRESET = 'iridium-50m'
preset = get_preset(PRESET)
print(preset_table())
result = dry_run(preset)
if not result['match']:
    raise RuntimeError(f"parameter count mismatch: {result}; do not train until this reconciles")

cuda: Tesla T4, 15.6 GB, inference dtype fp32 (CUDA 13.0, sm_75)
GPU: Tesla T4
preset        pri       params    body  embed   tokens tok/body    T4 h  P100 h  v5e h  status
----------------------------------------------------------------------------------------------
chat-34m        1   36,038,426   30.1M   2.1M     328M       11     4.1     3.6    0.2  verified in theory
chat-36m        1   46,180,869   35.8M   5.9M     716M       20    14.3    12.5    0.6  verified in theory
50m             1   52,719,370   45.1M   3.2M     328M        7     7.0     6.1    0.3  verified in theory
iridium-50m     1   60,273,095   50.4M   7.9M    5000M       99   102.5    89.3    4.2  verified in theory
100m            1  108,304,153   94.5M   8.4M     655M        7    21.1    18.4    0.9  verified in theory
chat-100m       1  108,304,153   94.5M   8.4M     655M        7    21.1    18.4    0.9  verified in theory
500m            1  497,686,932  449.5M  33.6M    1966M        4   458.6   399.4   18.9  v

In [4]:
import re, shutil
from pathlib import Path

OUT_DIR = Path('/kaggle/working/iridium-runs')
RUN_DIR = OUT_DIR / PRESET
RUN_DIR.mkdir(parents=True, exist_ok=True)
INPUTS = Path('/kaggle/input')

def attached(pattern):
    return sorted(INPUTS.rglob(pattern)) if INPUTS.is_dir() else []

def round_of(path):
    found = re.search(r'-round(\d+)\.pt$', path.name)
    return int(found.group(1)) if found else -1

data_marks = attached(f'{PRESET}/prepared.json')
DATA_DIR = data_marks[0].parent.parent if data_marks else OUT_DIR / 'data'
print('data:', DATA_DIR, '(attached)' if data_marks else '(not attached)')

# Copy only the newest checkpoint: inputs are read-only, and disk is limited.
if not list(RUN_DIR.glob('*-round*.pt')):
    earlier = max(attached(f'{PRESET}-round*.pt'), key=round_of, default=None)
    if earlier is not None:
        print('resuming from', earlier)
        shutil.copy2(earlier, RUN_DIR / earlier.name)
    else:
        print('no earlier checkpoint attached: this run starts from step 0')

data: /kaggle/working/iridium-runs/data (not attached)
no earlier checkpoint attached: this run starts from step 0


In [ ]:
from iridium.training.prepare import prepare

if not (DATA_DIR / PRESET / 'prepared.json').exists():
    if HAS_GPU:
        print('WARNING: preparing data with a GPU attached spends GPU quota. A CPU session is free.')
    prepare(preset, DATA_DIR)
    print('data written to', DATA_DIR)
if not HAS_GPU:
    print('Data is ready. Saved as this version\'s output: attach it and rerun with a GPU.')

In [ ]:
from iridium.training.run_preset import train_preset

SESSION_LIMIT_HOURS = 11.0     # Kaggle kills a session at 12 h and may lose its output

checkpoint = None
if HAS_GPU:
    used = (time.monotonic() - SESSION_STARTED) / 3600
    checkpoint = train_preset(
        preset, device=DEVICE, out=str(OUT_DIR), data=str(DATA_DIR), resume='auto', seed=0,
        time_budget_hours=SESSION_LIMIT_HOURS - used, keep_rounds=1,
    )
    print('newest checkpoint:', checkpoint)